# Setup GTEx data
## Setup raw data from GTEx in preparation for batch effect comparison
### Author: Martin Loza
### Date: 25/12/18

After selecting the window and genes of interest, we will setup gene pairs for co-expression analysis with ENCODE and GTEx data.

The first step is to confirm that there are not big batch effects and setup genes of interest. In this case we will focus only in genes with a related ENSEMBL ID

In [23]:
# Change R language to English
Sys.setenv(LANGUAGE = "en")

# Init
suppressPackageStartupMessages({
    library(dplyr)
    library(stringr)
    library(ggplot2)
    library(patchwork)
})

# Local variables 
seed = 777
date = "251218"

# Define colors for strand plots
red = "#E41A1C"
blue = "#090a0bff"
# Define colors for gene types
green = "#4DAF4A"
purple = "#984EA3"
text_size = 18
width = 18.6
dot_size = 4
line_size = 1.5
dpi = 300

gtex_dir = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/GTEx/"
gene_pairs_file = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Annotated_ncRNA_PCG_pairs/human_unique_lncRNA_TF_pairs_10000bp_251215.tsv"
out_dir = paste0(gtex_dir,"/normalized/")

# Local Functions


### Load and setup ENCODE data and metadata

In [24]:
rna_seq_files

[1] "GTEx_Analysis_v10_RNASeQCv2.4.2_gene_median_tpm_251218.gct"

In [25]:
# Get the RNA-seq file name. In this case, there is only one file with all the data in columns
rna_seq_files <- list.files(paste0(gtex_dir,"/raw/"))
# Load the data 
rna_seq_data <- read.table(paste0(gtex_dir,"/raw/",rna_seq_files), header = TRUE, sep = "\t", stringsAsFactors = FALSE, check.names = FALSE,skip = 2)
dim(rna_seq_data)

cat("Number of tissues (samples): ", ncol(rna_seq_data)-2, "\n")

[1] 59033    70

Number of tissues (samples):  68 


In [26]:
head(rna_seq_data,3)

,Name,Description,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta,Artery_Coronary,Artery_Tibial,Bladder,Brain_Amygdala,⋯,Spleen,Stomach,Stomach_Mixed_Cell,Stomach_Mucosa,Stomach_Muscularis,Testis,Thyroid,Uterus,Vagina,Whole_Blood
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,ENSG00000223972.5,DDX11L1,0.00000,0.00000,0.00000,0.0000,0.00000,0.00000,0.00000,0.00000,⋯,0.00000,0.00000,0.00000,0.00000,0.00000,0.167751,0.00000,0.0000,0.00000,0.00000
2,ENSG00000227232.5,WASH7P,3.99789,3.17815,2.67308,4.0708,3.87547,3.62625,5.05094,1.45902,⋯,6.04259,3.04117,3.32897,2.87374,4.08568,4.542470,6.31791,7.0687,5.74187,2.89503
3,ENSG00000278267.1,MIR6859-1,0.00000,0.00000,0.00000,0.0000,0.00000,0.00000,0.00000,0.00000,⋯,0.00000,0.00000,0.00000,0.00000,0.00000,0.000000,0.00000,0.0000,0.00000,0.00000


In [27]:
# Select genes with ENSG IDs only
rna_seq_data <- rna_seq_data %>%
        filter(str_detect(Name, pattern = "^ENSG"))

# Remove the version information of the ENSG IDs. This is located after the dot in the Name column
rna_seq_data <- rna_seq_data %>%
        mutate(Name = str_replace(Name, pattern = "\\..*$", replacement = ""))

cat("Number of genes with ENSG IDs: ", nrow(rna_seq_data), "\n")
head(rna_seq_data,3)

Number of genes with ENSG IDs:  59033 


,Name,Description,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta,Artery_Coronary,Artery_Tibial,Bladder,Brain_Amygdala,⋯,Spleen,Stomach,Stomach_Mixed_Cell,Stomach_Mucosa,Stomach_Muscularis,Testis,Thyroid,Uterus,Vagina,Whole_Blood
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,ENSG00000223972,DDX11L1,0.00000,0.00000,0.00000,0.0000,0.00000,0.00000,0.00000,0.00000,⋯,0.00000,0.00000,0.00000,0.00000,0.00000,0.167751,0.00000,0.0000,0.00000,0.00000
2,ENSG00000227232,WASH7P,3.99789,3.17815,2.67308,4.0708,3.87547,3.62625,5.05094,1.45902,⋯,6.04259,3.04117,3.32897,2.87374,4.08568,4.542470,6.31791,7.0687,5.74187,2.89503
3,ENSG00000278267,MIR6859-1,0.00000,0.00000,0.00000,0.0000,0.00000,0.00000,0.00000,0.00000,⋯,0.00000,0.00000,0.00000,0.00000,0.00000,0.000000,0.00000,0.0000,0.00000,0.00000


Let's create a log normalized data to compare its effect in the PCA distribution

In [28]:
# log normalized the TPM values
rna_seq_data_normalized <- rna_seq_data %>%  
    mutate(across(.cols = -c(Name, Description), .fns = ~log1p(.x)))

head(rna_seq_data_normalized,3)

,Name,Description,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta,Artery_Coronary,Artery_Tibial,Bladder,Brain_Amygdala,⋯,Spleen,Stomach,Stomach_Mixed_Cell,Stomach_Mucosa,Stomach_Muscularis,Testis,Thyroid,Uterus,Vagina,Whole_Blood
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,ENSG00000223972,DDX11L1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000000,⋯,0.000000,0.000000,0.00000,0.00000,0.000000,0.1550797,0.000000,0.000000,0.000000,0.000000
2,ENSG00000227232,WASH7P,1.609016,1.429869,1.301031,1.623499,1.584217,1.531747,1.800214,0.8997629,⋯,1.951976,1.396534,1.46533,1.35422,1.626429,1.7124402,1.990325,2.087992,1.908337,1.359701
3,ENSG00000278267,MIR6859-1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000000,⋯,0.000000,0.000000,0.00000,0.00000,0.000000,0.0000000,0.000000,0.000000,0.000000,0.000000


In [30]:
# Save the data and normalized data
out_file <- file.path(out_dir, paste0("raw_tpm_", date, ".tsv"))
write.table(rna_seq_data, file = out_file, sep = "\t", quote = FALSE, row.names = FALSE)

out_file <- file.path(out_dir, paste0("log_normalized_tpm_", date, ".tsv"))
write.table(rna_seq_data_normalized, file = out_file, sep = "\t", quote = FALSE, row.names = FALSE)